In [ ]:
# To run this code you need to install the following dependencies:
# pip install google-genai

!pip install -U -q "google"
!pip install -U -q "google.genai"

    opencv-python (>=3.) ; extra == 'all'
                  ~~~~^
    opencv-python (>=3.) ; extra == 'all'
                  ~~~~^
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
asyncer 0.0.2 requires anyio<4.0.0,>=3.4.0, but you have anyio 4.9.0 which is incompatible.
chainlit 0.6.1 requires aiofiles<24.0.0,>=23.1.0, but you have aiofiles 24.1.0 which is incompatible.
chainlit 0.6.1 requires fastapi<0.98.0,>=0.97.0, but you have fastapi 0.115.6 which is incompatible.
chainlit 0.6.1 requires openai<0.28.0,>=0.27.7, but you have openai 1.84.0 which is incompatible.
chainlit 0.6.1 requires pydantic<2.0.0,>=1.10.8, but you have pydantic 2.10.3 which is incompatible.
embedchain 0.0.65 requires langchain<0.0.280,>=0.0.279, but you have langchain 0.3.7 which is incompatible.
embedchain 0.0.65 requires openai<0.28.0,>=0.27.5, but you have openai 1.84.0 which is inco

In [4]:

import re
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv
from time import time

# Load environment variables from .env file
load_dotenv()
# Ensure the GEMINI_API_KEY environment variable is set
if not os.environ.get("GEMINI_API_KEY"):
    raise ValueError("GEMINI_API_KEY environment variable is not set. Please set it in your .env file.")
else:
    print("GEMINI_API_KEY is set.")


GEMINI_API_KEY is set.


# LLM Preprocessing

For the `file_patterns`, we will search for matching files in the `sanchaya` directory and process each file using the `do_llm_preprocess` function.

## Preprocessing Sanskrit Texts with LLMs

The `do_llm_preprocess` function processes Sanskrit literary works using Google's Gemini API. Here's what it does:

1. **Input Processing**: Takes a Sanskrit text file and checks if it exists
2. **Output Management**: Creates a unique output filename based on the input file . The output is saved in the `./llm_preprocessed/{file}_llm_pp.md`
3. **Text Splitting**: Divides the text into verses using "॥" as a delimiter
4. **Checkpointing**: Implements a checkpoint system to resume processing if interrupted
5. **API Integration**: Uses Google's Gemini API with a specified model (gemini-2.5-flash-preview)
6. **Batch Processing**: Processes verses in batches of 50 to manage API requests efficiently
7. **Time Tracking**: Records processing time in HH:MM:SS format for monitoring

The function is designed to work with a variety of Sanskrit texts including epics, poetry, and philosophical works. The pattern matching functionality helps identify specific works in the repository for targeted processing.

The `llm_pp.md` files have this template for each verse of the input file:

```markdown
## Verse 1.1

### Original Sloka
१.१ ॰प्रणिपत्य एकमनेकं कं सत्यां देवतां परं ब्रह्म्
१.१ आर्यभटस्त्रीणि ॰गदति गणितं कालक्रियां गोलम्

### Padacheda:
प्रणिपत्य एकमनेकम् कम् सत्यम् देवताम् परम् ब्रह्म् आर्यभटस्त्रीणि गदति गणितम् कालक्रियम् गोलम् ।

### Patterns म् + स्वर:  
|verse|म्+स्वरः|instances | count |
|---|---|---|---|
|1.1|आ|ब्रह्म् आर्यभटस्त्रीणि | 1|

```

The `padacheda` section provides a breakdown of the verse into its constituent parts, while the `Patterns म् + स्वर:` section lists instances of  `म् + स्वर:` patterns found in the padacheda output. These patterns are useful for linguistic analysis and further processing.



In [5]:
# these are the regex patterns for files from sanchaya to process
# these are carefully chosen so each pattern yields a unique file
# also if the special chars are stripeped, the name is still recognizable
file_patterns = [
    'raghuvamsham', 'kumarasambhava', 'kiratarjuniya',
    'shishupala', 'naishadiya',
    'gita.*govinda' , 'madhuravijaya',
    'meghadutam.txt', 'ratnavali', 
    # 'valmiki-ramayana.txt' , '^.*MB.*BORI.*$',
    "ashtanga.*hrud",
    "aryabhatiya.txt",
    "tarkasangraha",
    "udayana",
    "bharata.*natya",
    "sahrudaya",
    "gaarg",
    "shakuntalam",
    "mrcc.*dev",
    "yougandharayanam.txt",
    "bhasa.*svapna",
    "mudra.*raks",
    # anargha by murari
]

In [ ]:
# Function to preprocess the LLM input for a given Sanskrit work file


def do_llm_preprocess(
        sanskrit_work_file_name="../../../sanchaya/Kavyas, Kavya shastra/jayadeva, gita govinda.txt",
        dry_check=False,
    ):
    """Perform padacheda on the given verse file and generate content using Google GenAI."""
    # Check if the input file exists
    if not os.path.exists(sanskrit_work_file_name):
        print(f"Input file {sanskrit_work_file_name} does not exist.")
        return

    # output_file is file name part of the verse_file_name parameter
    output_file_sans_ext = sanskrit_work_file_name.split("/")[-1].split(".")[0]
    #replace consecutive non-alphanumeric characters with underscore
    output_file_sans_ext = re.sub(r'\W+', '_', output_file_sans_ext)
    # output_file = f"{output_file_sans_ext}_padacheda_output.md"
    output_file = f"llm_pre_processed/{output_file_sans_ext}_llm_pp.md"

    # return if the output file already exists
    if os.path.exists(output_file):
        print(f"Output file {output_file} already exists. Skipping processing.\n")
        return

    # Print the output file path
    print("\nStarting padacheda processing...")
    print(f"Input verse file: {sanskrit_work_file_name}")
    print("This may take a while depending on the size of the verse file.")
    print("Please wait...")
    # return

    ## read the verse file into a string
    with open(sanskrit_work_file_name, "r", encoding="utf-8") as file:
        verse_content = file.read()
        print("Verse content read successfully.")


    verses = verse_content.split("॥")  # Split the content into verses based on '॥'
    verses = [verse.strip() for verse in verses if verse.strip()]  # Clean up any empty strings
    print(f"Total verses found: {len(verses)}")

    if ( dry_check ): 
        print(f"Dry run mode enabled. Not processing verses. Total verses: {len(verses)}")
        print("Exiting without processing.")
        return
    # return

    checkpoint_dir = ".checkpoint"
    if not os.path.exists(checkpoint_dir): os.makedirs(checkpoint_dir)
    checkpoint_file = os.path.join(checkpoint_dir, f"{output_file_sans_ext}.checkpoint")

    with open(output_file, "a", encoding="utf-8") as fh:
        print(f"\n### Padacheda Output for {sanskrit_work_file_name} ###\n", file=fh)
    print(f"Output file initialized: {output_file}")

    
    # Check if the checkpoint file exists
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r", encoding="utf-8") as checkpoint_fh:
            start_count = int(checkpoint_fh.read().strip()) + 50
            print(f"Resuming from chunk: {start_count}")
    else:
        start_count = 0

    start_time = time()
    def time_since_start(): return time() - start_time
    def hhmmss(seconds):
        """Convert seconds to HH:MM:SS format."""
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        seconds = int(seconds % 60)
        return f"{hours:02}:{minutes:02}:{seconds:02}"
    def hms_since_start():
        """Get the elapsed time since the start in HH:MM:SS format."""
        return hhmmss(time_since_start())

    client = genai.Client(
        api_key=os.environ.get("GEMINI_API_KEY"),
    )
    model = "gemini-2.5-flash-preview-05-20"

    for verse_chunk in range(start_count,len(verses),50):
        n_verse_chunk = verses[verse_chunk:verse_chunk+50]
        print(f"{hms_since_start()} Processing {len(n_verse_chunk)} verses) : {n_verse_chunk[0:1]} ... {n_verse_chunk[-1:]} ")

        contents = [
            types.Content(
                role="user",
                parts=[
                    types.Part.from_text(text=f"""{n_verse_chunk} """),
                ],
            ),
            types.Content(
                role="model",
                parts=[
                    types.Part.from_text(text="""**Refining the Approach**

    I'm currently focused on the challenge of identifying and counting \"म् + स्वरः\" instances across word boundaries within each verse.  I've refined the approach to include verse-by-verse processing, starting with `padacheda` to separate the words. I'm now developing the logic to effectively pinpoint consecutive words that meet the \"म् + स्वरः\" criteria.


    **Pinpointing Instances**

    I've successfully processed the first verse and extracted the required information. The breakdown is as follows: `इदानीम् अपि` is the one instance of `म् + स्वरः` detected across word boundaries in verse 7.  The next verse is currently being parsed to find more instances.


    **Calculating Across Verses**

    I've now successfully processed two verses, identifying instances of \"म् + स्वरः\" across word boundaries. Verse 8 yields one instance: `कथम् अन्ये`. I'm systematically analyzing each verse to build the final list of \"म् + स्वरः\" instances, meticulously cross-referencing with the specified criteria.


    **Expanding the Analysis**

    I've just finished analyzing verses 9, 10, and 11. Verse 9 yielded no instances of \"म् + स्वरः\" across word boundaries. However, I found one such instance in verse 10: `वाचाम् आचान्तामृतसंपदाम्`.  I'm now collating the final results from all the verses to prepare the data for the markdown table.


    **Summarizing the Findings**

    I've completed the analysis of the verses. In verse 12, there are  two more instances of `म् + स्वरः`: `मन्दारमञ्जरी अस्यन्दिमकरन्दरसाब्धयः` and `कर्णामृतकवेः गिरः`. The `म् + स्वरः`  instances are as follows:  Verse 7: `इदानीम् अपि`; Verse 8: `कथम् अन्ये`; Verse 10: `वाचाम् आचान्तामृतसंपदाम्`; Verse 12: `मन्दारमञ्जरी अस्यन्दिमकरन्दरसाब्धयः` and `कर्णामृतकवेः गिरः`. The results are ready to be compiled into the markdown table.


    **Consolidating the Results**

    Now that I've analyzed all the verses, I've compiled the final count of \"म् + स्वरः\" instances. The breakdown, confirming the results from previous stages, is: Verse 7: `इदानीम् अपि`; Verse 8: `कथम् अन्ये`; Verse 10: `वाचाम् आचान्तामृतसंपदाम्`.  I'm ready to present the table to the user.


    **Summarizing the Analysis**

    I've just finished the analysis of all the verses. I've found that in Verse 12, there are instances of `म् + स्वरः`, specifically `मन्दारमञ्जरी अस्यन्दिमकरन्दरसाब्धयः` and `कर्णामृतकवेः गिरः`.  The complete list of instances is: Verse 7: `इदानीम् अपि`; Verse 8: `कथम् अन्ये`; Verse 10: `वाचाम् आचान्तामृतसंपदाम्`; Verse 12: `मन्दारमञ्जरी अस्यन्दिमकरन्दरसाब्धयः` and `कर्णामृतकवेः गिरः`.  I'm compiling the results for the markdown table.


    **Analyzing Verse Instances**

    I've completed a thorough analysis of all the verses, meticulously identifying and counting the instances of \"म् + स्वरः\" across word boundaries. Here's a quick summary: Verse 7 has one instance, `इदानीम् अपि`; Verse 8 shows one, `कथम् अन्ये`; Verse 10, `वाचाम् आचान्तामृतसंपदाम्`. Additionally, Verse 15 gave  `तम् अपरम्` and finally, verse 12 has `मन्दारमञ्जरी अस्यन्दिमकरन्दरसाब्धयः` and `कर्णामृतकवेः गिरः`. All results are compiled for the markdown table.


    **Compiling Final Results**

    I've meticulously processed each verse and identified the following \"म् + स्वरः\" instances across word boundaries: Verse 7: `इदानीम् अपि`; Verse 8: `कथम् अन्ये`; Verse 10: `वाचाम् आचान्तामृतसंपदाम्`; Verse 12: `मन्दारमञ्जरी अस्यन्दिमकरन्दरसाब्धयः` and `कर्णामृतकवेः गिरः`; and Verse 15: `तम् अपरम्`. The results are now ready to be compiled into the markdown table format.


    **Revising the Counts**

    I'm now refining the count of \"म् + स्वरः\" instances. The initial count needs some correction for the verses analyzed. Upon re-analysis, I confirmed verse 7 has the `इदानीम् अपि` instance, verse 8 has `कथम् अन्ये`. Additionally, verse 10 has `वाचाम् आचान्तामृतसंपदाम्`. No instance has been found for the verses 9, 11, 13, 14, 16 and 17. Moreover, verse 12 has `मन्दारमञ्जरी अस्यन्दिमकरन्दरसाब्धयः` and `कर्णामृतकवेः गिरः` and verse 15 has `तम् अपरम्`. I'm verifying all instances before compiling the markdown table.


    **Analyzing and Summarizing**

    I've successfully identified the instances of \"म् + स्वरः\" across verses. The instances are: Verse 7: `इदानीम् अपि`, Verse 8: `कथम् अन्ये`, Verse 10: `वाचाम् आचान्तामृतसंपदाम्`, Verse 12: `मन्दारमञ्जरी अस्यन्दिमकरन्दरसाब्धयः` and `कर्णामृतकवेः गिरः`, and Verse 15: `तम् अपरम्`.  I'm ready to present the final findings.


    **Summarizing the Analysis**

    I've meticulously processed each verse and identified the following \"म् + स्वरः\" instances across word boundaries: Verse 7: `इदानीम् अपि`; Verse 8: `कथम् अन्ये`; Verse 10: `वाचाम् आचान्तामृतसंपदाम्`; and Verse 15: `तम् अपरम्`.  All verses have been analyzed, and I'm now structuring the results for the markdown table.


    **Confirming Verse Analysis**

    I've re-examined each verse, meticulously verifying the \"म् + स्वरः\" instances. The verses with identified instances and their counts have been double-checked for accuracy before compiling the markdown table.


    """),
                    
                ],
            ),
            types.Content(
                role="user",
                parts=[
                    types.Part.from_text(text="""INSERT_INPUT_HERE"""),
                ],
            ),
        ]
        generate_content_config = types.GenerateContentConfig(
            # thinking_config = types.ThinkingConfig(
            #     thinking_budget=3000,
            # ),
            response_mime_type="text/plain",
            system_instruction=[
                types.Part.from_text(text="""You are an expert in the Sanskrit language. 
    You are here to assist  count \"म्+[स्वरः]\" sounds across consecutive word boundaries.  
    We are not interested in other sound patterns 
    The steps would be for each verse do a पदच्छेद,  
    - list the instances of two consecutive padas where the first ends in म् and the next pada starts with a स्वरः   
    - count the instances of म्+स्वरः for each svara
    - provide response as in the example below in raw markdown

    Validate you are counting is limited to all occurrences of   \"म्+[स्वरः]\" sounds across consecutive word boundaries. 

    Example :
    ##  Verse 15

    ### Original Sloka 
    स्तुमस्तमपरं व्यासं गङ्गाधर महाकविम् 
    नाटकच्छद्मना दृष्टां यश्चक्रे भारतीं कथाम् ॥ १५ ॥

    ### Padacheda:
    स्तुमः तम् अपरम् व्यासम् गङ्गाधर महाकविम् ।
    नाटक छद्मना दृष्टाम् यः चक्रे भारतीम् कथाम् ॥

    ### Patterns म् + स्वर:  
    |verse|म्+स्वरः|instances | count |
    |17|अ| तम् अपरम् , | 1|

    ------
    """),
            ],
        )


        cntr=0
        print(f"\t {hms_since_start()} Processing chunk: {verse_chunk:03d}/{len(verses):4d}")
        for chunk in client.models.generate_content_stream(
            model=model,
            contents=contents,
            config=generate_content_config,
        ):
            # if chunk.error:
            #     print(f"Error: {chunk.error.message}")
            # else:
            #     print("Chunk received successfully.")
            # print(chunk.text[:100], end="")

            # save the chunk to a file
            with open(f"{output_file}", "a", encoding="utf-8") as output_fh:
                if chunk.text:
                    output_fh.write(chunk.text)
            if (cntr % 10 == 0):
                print(f"\t\tProcessed {cntr} chunks so far...")
            cntr += 1
        # Overwrite the checkpoint file with the current processed chunk
        with open(checkpoint_file, "w", encoding="utf-8") as checkpoint_fh:
            checkpoint_fh.write(f"{verse_chunk}\n")

    print(f"{hms_since_start()} Padacheda processing completed. Output saved to {output_file}")


# if __name__ == "__main__":
#     works_of_interest = [
#         "kalidasa, kumarasambhava.txt",
#         "kalidasa, raghuvamsham.txt",
#         "bharavi, kiratarjuniya.txt",
#         "magha, shishupala vadha.txt",
#         "shriharsha, naishadiya charitam.txt",
#         "kalidasa, meghadutam.txt",
#         "gangadevi, madhuravijaya.txt",
#         "jayadeva, gita govinda.txt",
#     ]

# for work in works_of_interest:
#     verse_file_name = f"../../../sanchaya/Kavyas, Kavya shastra/{work}"
#     print(f"\nProcessing work: {work}")
#     do_padacheda(verse_file_name=verse_file_name)


In [7]:


def list_files_with_regex_patterns(patterns, directory) : 
    """List files in the given directory that match any of the provided patterns."""
    import fnmatch
    matching_files = []
    for root, dirs, files in os.walk(directory):
        for filename in files:
            for pattern in patterns:
                if re.search(pattern, filename, re.IGNORECASE):
                    # If a match is found, add the full path to the matching_files list     
                    matching_files.append(os.path.join(root, filename))
    return matching_files

sanchaya_dir = "../../../sanchaya/"
# kavyas_dir = f"{sanchaya_dir}Kavyas, Kavya shastra/"

matching_files = list_files_with_regex_patterns(file_patterns, sanchaya_dir)
print(f"\nFound {len(matching_files)} matching files:")
for ix, file in enumerate(matching_files):
    print(f"{ix+1:02d} : {file}")
    do_llm_preprocess(sanskrit_work_file_name=file, dry_check=not True)
print("\nAll specified works have been processed.")


Found 21 matching files:
01 : ../../../sanchaya/Nyaya Vaisheshika/udayana, nyaya kusumanjali.txt
Output file llm_pre_processed/udayana_nyaya_kusumanjali_llm_pp.md already exists. Skipping processing.

02 : ../../../sanchaya/Nyaya Vaisheshika/tarkasangraha मूलम्.txt
Output file llm_pre_processed/tarkasangraha_म_लम__llm_pp.md already exists. Skipping processing.

03 : ../../../sanchaya/Jyotisham/aryabhatiya.txt
Output file llm_pre_processed/aryabhatiya_llm_pp.md already exists. Skipping processing.

04 : ../../../sanchaya/Jyotisham/vrrddha-gaargiiya-jyotisham.txt
Output file llm_pre_processed/vrrddha_gaargiiya_jyotisham_llm_pp.md already exists. Skipping processing.

05 : ../../../sanchaya/Medicine/ashtanga hrudaya.txt
Output file llm_pre_processed/ashtanga_hrudaya_llm_pp.md already exists. Skipping processing.

06 : ../../../sanchaya/Kavyas, Kavya shastra/kalidasa, meghadutam.txt
Output file llm_pre_processed/kalidasa_meghadutam_llm_pp.md already exists. Skipping processing.

07 : ../.